In [ ]:
# Importamos la librería requests para hacer peticiones HTTP a la API
import requests
# Importamos json para manejar la lectura y escritura de archivos JSON
import json
# Importamos JSON de IPython.display para mostrar estructuras JSON en Jupyter (opcional en notebooks)
from IPython.display import JSON
# Importamos nuestras llaves/credenciales desde el archivo keys.py
from keys import *

In [ ]:
# Definimos la URL de la API con el aeropuerto EDDB (Berlín-Brandeburgo) y un rango de horas (08:00 a 20:00)
url = "https://aerodatabox.p.rapidapi.com/flights/airports/icao/MMMX/2025-09-01T08:00/2025-09-01T20:00"

# Parámetros opcionales para filtrar los vuelos
querystring = {"withLeg":"true","direction":"Arrival","withCancelled":"true","withCodeshared":"true","withCargo":"true","withPrivate":"true","withLocation":"false"}

# Encabezados necesarios para la autenticación con RapidAPI
headers = {
    'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
    'x-rapidapi-key': flights_key
    }

# Realizamos la petición GET a la API con la URL, los headers y los parámetros
response = requests.request("GET", url, headers=headers, params=querystring)

In [ ]:
# Si la respuesta es exitosa (código 200)
if response.status_code == 200:
        # Convertimos la respuesta JSON en un diccionario de Python
    new_data = response.json()

    try:
     # Intentamos abrir el archivo "data.json" para cargar datos existentes
        with open("data.json", "r") as json_file:
            existing_data = json.load(json_file)
    except (FileNotFoundError, json.decoder.JSONDecodeError):
        # Si el archivo no existe o está vacío/corrupto, creamos una lista vacía
        existing_data = []

    # Agregamos la nueva información de vuelos al archivo existente
    existing_data.append(new_data)

    # Guardamos el archivo "data.json" con los datos actualizados
    with open("data.json", "w") as json_file:
        json.dump(existing_data, json_file, indent=4)
        print("Data appended to data.json file.")
else:
    # Guardamos el archivo "data.json" con los datos actualizados
    print("Failed to retrieve data from the API. Status code:", response.status_code)

Data appended to data.json file.


In [ ]:
# Guardamos nuevamente la respuesta en una variable para trabajar con ella
flight_arrivals = response.json()

#Exploración de los datos (no imprime nada porque no usas print)
for flight in flight_arrivals:
    flight_arrivals['arrivals'][0]['arrival']['scheduledTime']
    flight_arrivals['arrivals'][0]['number']
    flight_arrivals['arrivals'][0]['arrival']['terminal']
    flight_arrivals['arrivals'][0]['departure']['airport']['name']
    flight_arrivals['arrivals'][0]['aircraft']['model']
    flight_arrivals['arrivals'][0]['airline']['name']

In [ ]:
# Importamos pandas para manipulación de datos
import pandas as pd
# Importamos nan de numpy para representar valores faltantes (NaN)
from numpy import nan

# Guardamos otra vez la respuesta como JSON
flight_arrivals = response.json()

# Definimos una función para extraer información relevante de un vuelo
def flight_extraction(flights):
    # Usamos pd.json_normalize para convertir el JSON de un vuelo en DataFrame
    flights_df = pd.json_normalize({
        'Scheduled_arrival_time': flights['arrival']['scheduledTime'],
        'flight_number': flights['number'],
        'from' : flights['departure']['airport']['name'],
        'airline' : flights['airline']['name'],
        'aircraft' : flights['aircraft']['model']
    })
    return(flights_df) # Devolvemos el DataFrame

# Concatenamos todos los vuelos (arrivals) en un solo DataFrame
flight_arrivals = pd.concat(
    [flight_extraction(flight) for flight in flight_arrivals['arrivals']]
)

# Mostramos las primeras filas del DataFrame
flight_arrivals.head()

,flight_number,from,airline,aircraft,Scheduled_arrival_time.utc,Scheduled_arrival_time.local
0,UA 1024,Houston,United,Boeing 737-700,2025-09-01 14:35Z,2025-09-01 08:35-06:00
0,AM 505,Cancún,Aeromexico,Boeing 737-800,2025-09-01 14:30Z,2025-09-01 08:30-06:00
0,AM 905,Monterrey,Aeromexico,Boeing 737-900,2025-09-01 14:30Z,2025-09-01 08:30-06:00
0,AM 205,Guadalajara,Aeromexico,Boeing 737,2025-09-01 14:35Z,2025-09-01 08:35-06:00
0,VB 1355,Monterrey,VivaAerobus,Airbus A321 NEO,2025-09-01 14:40Z,2025-09-01 08:40-06:00
